# 04 — Experimentos con MLflow

Este notebook registra en MLflow los experimentos del TP1 (ya completados) y sirve de
base para los experimentos del TF1 (Optuna, nuevos modelos).

**No re-evalúa el test set** — solo loguea los resultados OOF ya calculados.

Para ver la UI de MLflow, correr en la terminal (desde la raíz del proyecto):
```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db
```
y abrir http://127.0.0.1:5000

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import mlflow
from src.mlflow_utils import (
    init_run,
    log_cv_results,
    log_threshold_metrics,
    log_ablation,
    log_model_artifact,
    log_figure,
    setup_tracking,
)
from src import config as C

# Configurar tracking con el backend SQLite predeterminado del proyecto
setup_tracking()

# Cargar números oficiales
with open(ROOT / 'reports' / 'numeros_oficiales.json', encoding='utf-8') as f:
    nums = json.load(f)

print('MLflow version:', mlflow.__version__)
print('Tracking URI:', mlflow.get_tracking_uri())

MLflow version: 3.18.0
Tracking URI: C:/Users/rivas/Desktop/2026/2026-2/Data Mining Tools/TP1_DataMining/mlruns


## 1. Registrar los modelos del TP1 (resultados OOF)

Cada modelo del TP1 se registra como un run separado dentro del experimento `dropout_tp1`.
No re-entrenamos — solo logueamos los números ya calculados.

In [4]:
db_path = (ROOT / "mlflow.db").as_posix()
mlflow.set_tracking_uri(f"sqlite:///{db_path}")
print("Tracking URI:", mlflow.get_tracking_uri())

EXPERIMENT = 'dropout_tp1'

# Modelos a registrar con sus métricas OOF
modelos = [
    ('logreg',           nums['modelos_oof']['logreg']),
    ('hist_gb',          nums['modelos_oof']['hist_gb']),
    ('random_forest',    nums['modelos_oof']['random_forest']),
    ('regla_aprobadas0', nums['modelos_oof']['regla_aprobadas0']),
]

for nombre, metricas in modelos:
    with init_run(experiment=EXPERIMENT, run_name=nombre,
                    tags={'fase': 'tp1', 'modelo': nombre}) as run:
        log_cv_results(metricas)
        print(f'  Registrado: {nombre} (run_id={run.info.run_id[:8]}…)')

print('\n Todos los modelos TP1 registrados en MLflow.')

Tracking URI: sqlite:///C:/Users/rivas/Desktop/2026/2026-2/Data Mining Tools/TP1_DataMining/mlflow.db
  Registrado: logreg (run_id=de5e2a94…)
  Registrado: hist_gb (run_id=51dd458b…)
  Registrado: random_forest (run_id=5d6e7b3f…)
  Registrado: regla_aprobadas0 (run_id=64e7fcfe…)

 Todos los modelos TP1 registrados en MLflow.


## 2. Registrar el modelo ganador (LogReg) con artefactos completos

In [5]:
import joblib

model_path = ROOT / 'models' / 'dropout_logreg_v0.1-tp1.joblib'
meta_path  = ROOT / 'models' / 'dropout_logreg_v0.1-tp1.json'

pipe = joblib.load(model_path)
with open(meta_path, encoding='utf-8') as f:
    meta = json.load(f)

op = nums['test']['resultados']['punto_operacion']

with init_run(experiment=EXPERIMENT, run_name='logreg_ganador_tp1',
              tags={'fase': 'tp1', 'modelo': 'logreg', 'ganador': 'true'}) as run:

    # Métricas OOF
    log_cv_results(nums['modelos_oof']['logreg'])

    # Métricas test (solo informativo — no se usa para selección)
    mlflow.log_metric('test_pr_auc',  nums['test']['resultados']['pr_auc'][0])
    mlflow.log_metric('test_roc_auc', nums['test']['resultados']['roc_auc'][0])

    # Punto de operación
    log_threshold_metrics(
        threshold=op['umbral'],
        recall=op['recall'], precision=op['precision'], f1=op['f1'],
        tp=op['TP'], fp=op['FP'], fn=op['FN'], tn=op['TN'],
    )

    # Ablación
    log_ablation(nums['ablacion'])

    # Pipeline serializado + metadata
    log_model_artifact(pipe, model_name='logreg', threshold=op['umbral'], metadata=meta)

    # Figuras
    for fig in (ROOT / 'reports' / 'figures').glob('*.png'):
        log_figure(fig)

    print(f'✓ Run completo: run_id={run.info.run_id}')

c:\Users\rivas\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:440: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.4.0 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\rivas\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:440: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.4.0 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\rivas\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:440: InconsistentVersionWarning: Trying to unpickle estimato

✓ Run completo: run_id=623125aabb7e4853adc3a04e989ccc92


## 3. Comparar runs registrados

In [6]:
import pandas as pd

mlflow.set_experiment(EXPERIMENT)
runs_df = mlflow.search_runs(order_by=['metrics.cv_pr_auc_mean DESC'])

cols = ['tags.mlflow.runName', 'metrics.cv_pr_auc_mean', 'metrics.cv_roc_auc_mean',
        'metrics.threshold', 'metrics.op_recall', 'metrics.op_precision']
cols_exist = [c for c in cols if c in runs_df.columns]

print(runs_df[cols_exist].to_string(index=False))

tags.mlflow.runName  metrics.cv_pr_auc_mean  metrics.cv_roc_auc_mean  metrics.threshold  metrics.op_recall  metrics.op_precision
 logreg_ganador_tp1                0.860749                 0.903544            0.24229           0.862676              0.686275
             logreg                0.860749                 0.903544                NaN                NaN                   NaN
             logreg                0.860749                 0.903544                NaN                NaN                   NaN
            hist_gb                0.855360                 0.899176                NaN                NaN                   NaN
            hist_gb                0.855360                 0.899176                NaN                NaN                   NaN
      random_forest                0.844654                 0.893621                NaN                NaN                   NaN
      random_forest                0.844654                 0.893621                NaN          

## 4. Ver la UI de MLflow

Correr en la terminal **fuera de este notebook** (desde la raíz del proyecto):

```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db
```

Luego abrir: **http://127.0.0.1:5000**

> El servidor de MLflow es un proceso que queda corriendo — no ejecutar desde el notebook.